In [16]:
import numpy as np
from neurora.rdm_corr import rdm_correlation_spearman
from neurora.rsa_plot import plot_rdm
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import pandas as pd
import pingouin as pg
from neurora.stuff import smooth_1d
from scipy.stats import spearmanr, ttest_1samp, wilcoxon
from scipy.spatial import distance
from scipy.optimize import nnls
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from itertools import product
from mlxtend.evaluate import permutation_test

plt.rcParams['figure.dpi'] = 300

## Calculate EEG RDMs

In [ ]:
sub_ids = ['01', '02', '03', '04', '05', '06', '07', '09']

for sub_id in sub_ids:
    
    data = np.load('sub'+sub_id+'_afterica_eegdata_condavg.npy')
    data = np.average(data, axis=(0, 2))
    
    print(data.shape)
    
    data_20ms = np.zeros([64, 63, 100])
    for t in range(100):
        data_20ms[:, :, t] = np.average(data[:, :, t*20:t*20+20], axis=2)
    
    eegrdms = np.zeros([2, 100, 64, 64])
    # 2 measurements (amplitude or pattern) * 3 regions * 100 time-points * 64 * 64
    
    # based on amplitude
    print('Sub'+sub_id+' EEG RDMs based on amplitude')
    
    for t in tqdm(range(100)):
        for i in range(64):
            v1 = np.average(data_20ms[i, :, t])
            for j in range(64):
                if j > i:
                    v2 = np.average(data_20ms[j, :, t])
                    eegrdms[0, t, i, j] = np.abs(v1-v2)
                    eegrdms[0, t, j, i] = eegrdms[0, t, i, j]
    
    # based on pattern
    print('Sub'+sub_id+' EEG RDMs based on pattern')
    for i in range(63):
        data_20ms[:, i] = (data_20ms[:, i]-np.average(data_20ms[:, i]))/np.std(data_20ms[:, i])
    
    for t in tqdm(range(100)):
        for i in range(64):
            v1 = data_20ms[i, :, t].flatten()
            for j in range(64):
                if j > i:
                    v2 = data_20ms[j, :, t].flatten()
                    eegrdms[1, t, i, j] = 1 - pearsonr(v1, v2)[0]
                    eegrdms[1, t, j, i] = eegrdms[1, t, i, j]
                    
    np.save('sub'+sub_id+'_afterica_eegRDMs_allchls.npy', eegrdms)

## Correlation

In [ ]:
modelrdms = np.load('../model_rdms.npy')
print(modelrdms.shape)

sub_ids = ['01', '02', '03', '04', '05', '06', '07', '09']

for sub_id in sub_ids:

    corrs = np.zeros([2, 12, 100])
    eegrdms = np.load('sub'+sub_id+'_afterica_eegRDMs_allchls.npy')
    print(np.max(eegrdms[0]))
    for i in range(12):
        modelrdm = modelrdms[i]
        modelrdm_flat = modelrdm[np.triu_indices_from(modelrdm, k=1)]
        for t in range(100):
            for j in range(2):
                eegrdm = eegrdms[j, t]
                eegrdm_flat = eegrdm[np.triu_indices_from(eegrdm, k=1)]
                mean = np.mean(eegrdm_flat)
                std = np.std(eegrdm_flat)
                eegrdm_flat = (eegrdm_flat-mean)/std
                corrs[j, i, t] = spearmanr(modelrdm_flat, eegrdm_flat)[0]

    smooth_corrs = np.zeros([2, 12, 100])
    corrs = np.reshape(corrs, [1, 2, 12, 100])
    for i in range(2):
        for j in range(11):
            smooth_corrs[i, j] = smooth_1d(corrs[:, i, j], n=5)[0]
        
    np.save('rsa/rsa_corrs_sub'+sub_id+'_allchls.npy', smooth_corrs)

for sub_id in sub_ids:

    corrs = np.zeros([2, 12, 3, 100])
    eegrdms = np.load('sub'+sub_id+'_afterica_eegRDMs.npy')
    print(np.max(eegrdms[0]))
    for i in range(12):
        modelrdm = modelrdms[i]
        modelrdm_flat = modelrdm[np.triu_indices_from(modelrdm, k=1)]
        for t in range(100):
            for k in range(3):
                for j in range(2):
                    eegrdm = eegrdms[j, k, t]
                    eegrdm_flat = eegrdm[np.triu_indices_from(eegrdm, k=1)]
                    mean = np.mean(eegrdm_flat)
                    std = np.std(eegrdm_flat)
                    eegrdm_flat = (eegrdm_flat-mean)/std
                    corrs[j, i, k, t] = spearmanr(modelrdm_flat, eegrdm_flat)[0]

    smooth_corrs = np.zeros([2, 12, 3, 100])
    corrs = np.reshape(corrs, [1, 2, 12, 3, 100])
    for i in range(2):
        for j in range(12):
            for k in range(3):
                
                smooth_corrs[i, j, k] = smooth_1d(corrs[:, i, j, k], n=5)[0]
        
    np.save('rsa/rsa_corrs_sub'+sub_id+'.npy', smooth_corrs)

## Partial Correlation without Integration

In [ ]:
modelrdms = np.load('../model_rdms.npy')[[0, 1, 2, 3, 4, 5, 6, 9, 10, 11]]
print(modelrdms.shape)

sub_ids = ['01', '02', '03', '04', '05', '06', '07', '09']

for sub_id in sub_ids:

    corrs = np.zeros([2, 10, 100])

    triu_indices = np.triu_indices(64, k=1)
    modelrdms_flat = modelrdms[:, triu_indices[0], triu_indices[1]]

    print(sub_id)
    eegrdms = np.load('sub'+sub_id+'_afterica_eegRDMs_allchls.npy')
    for t in range(100):
        for j in range(2):
            eegrdm = eegrdms[j, t]
            eegrdm_flat = eegrdm[np.triu_indices_from(eegrdm, k=1)]
            mean = np.mean(eegrdm_flat)
            std = np.std(eegrdm_flat)
            eegrdm_flat = (eegrdm_flat-mean)/std
            for i in range(10):
                data = {
                    'y': eegrdm_flat,
                    'x': modelrdms_flat[i]
                }
                covars = {f'x{covar_i}': modelrdms_flat[covar_i] for covar_i in range(10) if covar_i != i}
                data.update(covars)
                df = pd.DataFrame(data)
                stats = pg.partial_corr(data=df, x='x', y='y', x_covar=list(covars.keys()),
                                        alternative='greater', method='spearman')
                corrs[j, i, t] = stats['r'][0]
    
    smooth_corrs = np.zeros([2, 10, 100])
    corrs = np.reshape(corrs, [1, 2, 10, 100])
    for i in range(2):
        for j in range(10):
            smooth_corrs[i, j] = smooth_1d(corrs[:, i, j], n=5)[0]
        
    np.save('rsa/rsa_partialcorrs_10_sub'+sub_id+'_allchls.npy', smooth_corrs)

## Partial Correlation with Integration

In [ ]:
modelrdms = np.load('../model_rdms.npy')[[7, 8]]
print(modelrdms.shape)

sub_ids = ['01', '02', '03', '04', '05', '06', '07', '09']

for sub_id in sub_ids:

    corrs = np.zeros([2, 2, 100])

    triu_indices = np.triu_indices(64, k=1)
    modelrdms_flat = modelrdms[:, triu_indices[0], triu_indices[1]]

    print(sub_id)
    eegrdms = np.load('sub'+sub_id+'_afterica_eegRDMs_allchls.npy')
    for t in range(100):
        for j in range(2):
            eegrdm = eegrdms[j, t]
            eegrdm_flat = eegrdm[np.triu_indices_from(eegrdm, k=1)]
            mean = np.mean(eegrdm_flat)
            std = np.std(eegrdm_flat)
            eegrdm_flat = (eegrdm_flat-mean)/std
            for i in range(2):
                data = {
                    'y': eegrdm_flat,
                    'x': modelrdms_flat[i]
                }
                covars = {f'x{covar_i}': modelrdms_flat[covar_i] for covar_i in range(2) if covar_i != i}
                data.update(covars)
                df = pd.DataFrame(data)
                stats = pg.partial_corr(data=df, x='x', y='y', x_covar=list(covars.keys()),
                                        alternative='greater', method='spearman')
                corrs[j, i, t] = stats['r'][0]
    
    smooth_corrs = np.zeros([2, 2, 100])
    corrs = np.reshape(corrs, [1, 2, 2, 100])
    for i in range(2):
        for j in range(2):
            smooth_corrs[i, j] = smooth_1d(corrs[:, i, j], n=5)[0]
        
    np.save('rsa/rsa_partialcorrs_integration_sub'+sub_id+'_allchls.npy', smooth_corrs)

In [5]:
modelrdms = np.load('../model_rdms.npy')
print(modelrdms.shape)

sub_ids = ['01', '02', '03', '04', '05', '06', '07', '09']

for sub_id in sub_ids:

    corrs = np.zeros([2, 12, 100])

    triu_indices = np.triu_indices(64, k=1)
    modelrdms_flat = modelrdms[:, triu_indices[0], triu_indices[1]]

    print(sub_id)
    eegrdms = np.load('sub'+sub_id+'_afterica_eegRDMs_allchls.npy')
    for t in range(100):
        for j in range(2):
            eegrdm = eegrdms[j, t]
            eegrdm_flat = eegrdm[np.triu_indices_from(eegrdm, k=1)]
            mean = np.mean(eegrdm_flat)
            std = np.std(eegrdm_flat)
            eegrdm_flat = (eegrdm_flat-mean)/std
            for i in range(12):
                data = {
                    'y': eegrdm_flat,
                    'x': modelrdms_flat[i]
                }
                covars = {f'x{covar_i}': modelrdms_flat[covar_i] for covar_i in range(12) if covar_i != i}
                data.update(covars)
                df = pd.DataFrame(data)
                stats = pg.partial_corr(data=df, x='x', y='y', x_covar=list(covars.keys()),
                                        alternative='greater', method='spearman')
                corrs[j, i, t] = stats['r'][0]
    
    smooth_corrs = np.zeros([2, 12, 100])
    corrs = np.reshape(corrs, [1, 2, 12, 100])
    for i in range(2):
        for j in range(12):
            smooth_corrs[i, j] = smooth_1d(corrs[:, i, j], n=5)[0]
        
    np.save('rsa/rsa_partialcorrs_12_sub'+sub_id+'_allchls.npy', smooth_corrs)